# PolicyRec v1.0 데이터 수집 파이프라인

이 노트북은 **청년·예비창업자·기업 맞춤형 AI 비서**를 만들기 위한 첫 번째 데이터 준비 노트북입니다.

목표는 간단합니다.

1. `data/raw`에 저장된 원본 JSON 파일을 읽습니다.
2. Bizinfo, K-Startup, Youthcenter 데이터 구조를 확인합니다.
3. 각 API의 원본 컬럼명을 바꾸지 않고 그대로 유지합니다.
4. 세 데이터를 하나의 CSV로 합칩니다.
5. 결과를 `data/clean/combined_raw_columns.csv`로 저장합니다.

중요한 점은 **정규화하지 않는다**는 것입니다.

예를 들어 `pblancNm`을 `title`로 바꾸지 않습니다. `pbanc_rcpt_bgng_dt`를 `apply_start`로 바꾸지도 않습니다. 원본 컬럼은 그대로 두고, 출처 확인용 컬럼만 앞에 추가합니다.

## 버전별 개발 계획

이 프로젝트는 한 번에 완성형 추천 서비스를 만드는 방식이 아니라, 데이터를 다루는 범위를 조금씩 넓히는 방식으로 진행합니다.

| 버전 | 목표 | 결과물 | 아직 하지 않는 것 |
| :--- | :--- | :--- | :--- |
| `v1` | API 호출 + 원본 그대로 통합 CSV 만들기 | `data/clean/combined_raw_columns.csv` | 컬럼명 통일, 날짜 변환, 첨부파일 분석 |
| `v1.n` | v1 기반 + 최소 정규화 | 공통 컬럼 CSV 후보 | 첨부파일 본문까지 포함한 분석 |
| `v2` | v1.n 기반 + 첨부파일 추가 | 목록 데이터 + 첨부파일 정보가 연결된 데이터 | 최종 추천 모델, Q&A 고도화 |

### v1: 지금 노트북의 범위

v1에서는 “원본을 잃지 않는 것”이 가장 중요합니다.

그래서 API에서 받은 컬럼명을 마음대로 바꾸지 않습니다. 예를 들어 Bizinfo의 `pblancNm`, K-Startup의 `intg_pbanc_biz_nm`, Youthcenter의 `plcyNm`은 모두 제목처럼 보이지만, v1에서는 이 세 컬럼을 억지로 `title` 하나로 합치지 않습니다.

대신 세 데이터를 세로로 붙이고, 어느 source에서 온 행인지 알 수 있도록 `_source`, `_source_name`, `_source_file`, `_source_row_number`만 추가합니다.

### v1.n: 다음 단계의 최소 정규화

v1.n에서는 추천과 검색에 자주 쓸 최소한의 공통 컬럼을 따로 만듭니다.

예상 작업은 아래와 같습니다.

- 컬럼명 통일: `pblancNm`, `intg_pbanc_biz_nm`, `plcyNm` 같은 제목 후보를 공통 컬럼으로 정리
- 날짜 형식 통일: `20260420`, `2026-04-20`, `2026-04-14 ~ 2026-05-06` 같은 값을 비교 가능한 형태로 변환
- 결측값 처리: source마다 없는 값은 빈칸, `없음`, `전국`, `연중`처럼 기준을 정해 처리
- 추천에 쓸 후보 컬럼 선별: 제목, 요약, 지역, 대상, 나이, 신청 기간, 상세 URL 등

여기서도 원본 CSV는 버리지 않습니다. v1 결과를 기준으로, 필요한 컬럼만 새로 정리한 파일을 추가로 만드는 방식이 안전합니다.

### v2: 첨부파일까지 확장

v2에서는 공고 목록 데이터만으로 부족한 정보를 첨부파일에서 보완합니다.

예를 들어 공고문 PDF, HWP, ZIP 안의 신청서류, RFP, 세부 조건 같은 정보가 여기에 해당합니다.

예상 작업은 아래와 같습니다.

- source별 첨부파일 위치 확인
- 공고 item과 첨부파일 연결
- PDF/HWP 등에서 텍스트 추출
- 첨부파일 본문을 요약하거나 검색에 쓸 수 있는 형태로 정리
- 필요하면 v1.n의 정규화 기준을 다시 조정

첨부파일은 형식이 다양하고 예외가 많기 때문에, v1에서 원본 목록 데이터를 먼저 안정화한 뒤 진행합니다.

## 전체 흐름

이 노트북은 아래 순서대로 진행됩니다.

1. 설정값과 helper 함수를 준비합니다.
2. 프로젝트 폴더 상태를 확인합니다.
3. 필요하면 API를 새로 호출합니다.
4. source별 최신 raw 파일을 찾습니다.
5. JSON 구조와 대표 item을 확인합니다.
6. 원본 컬럼 그대로 하나의 CSV로 통합합니다.
7. 통합 결과를 검증합니다.

바꿀 가능성이 있는 값은 대부분 첫 번째 코드 셀의 `사용자가 자주 바꿀 설정값` 영역에 모아 두었습니다.

In [1]:
# ============================================================
# 0. 기본 설정과 helper 함수 모음
# ============================================================
# 이 셀은 아래 모든 셀에서 공통으로 사용할 설정과 함수를 준비합니다.
# 값만 바꾸고 싶을 때는 먼저 "사용자가 자주 바꿀 설정값" 영역을 확인하세요.

from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    # Jupyter가 아닌 일반 Python에서 실행할 때도 검증할 수 있게 하는 대체 함수입니다.
    def display(value):
        print(value)


# ============================================================
# 사용자가 자주 바꿀 설정값
# ============================================================

# 이번 노트북에서 다룰 데이터 출처입니다.
SOURCES = ["biz", "kst", "youth"]

# source 코드를 사람이 읽기 좋은 이름으로 바꿔 보여줄 때 사용합니다.
SOURCE_LABELS = {
    "biz": "Bizinfo",
    "kst": "K-Startup",
    "youth": "Youthcenter",
}

# API를 새로 호출할 때 쓸 기본 옵션입니다.
PAGE = 1
PAGE_SIZE = 10

# False이면 API를 새로 호출하지 않고, 이미 저장된 data/raw 파일만 사용합니다.
# 처음 공부하거나 구조만 확인할 때는 False가 안전합니다.
RUN_FETCH = False

# True이면 기존 raw 파일이 있어도 다시 API를 호출합니다.
# API 호출 횟수나 키 제한이 있을 수 있으니 필요할 때만 True로 바꾸세요.
FORCE_FETCH = False

# raw 파일로 인정할 확장자입니다.
RAW_FILE_PATTERNS = ["*.json", "*.xml", "*.txt"]

# 화면에 너무 많이 출력되지 않도록 미리보기 길이를 조절합니다.
MAX_TOP_LEVEL_KEYS = 10
MAX_SAMPLE_ITEM_KEYS = 15
TEXT_PREVIEW_LENGTH = 120
PREVIEW_ROW_COUNT = 10

# CSV 저장 인코딩입니다.
# utf-8-sig는 Excel에서 한글이 깨지는 일을 줄여줍니다.
CSV_ENCODING = "utf-8-sig"

# 원본 컬럼과 구분하기 위해, 우리가 추가하는 관리용 컬럼은 앞에 _를 붙입니다.
METADATA_COLUMNS = ["_source", "_source_name", "_source_file", "_source_row_number"]

# source별 대표 item을 볼 때 우선 확인할 원본 필드입니다.
# 여기 있는 이름도 원본 API 컬럼명 그대로입니다.
PREVIEW_FIELDS = {
    "biz": [
        "pblancId",
        "pblancNm",
        "reqstBeginEndDe",
        "jrsdInsttNm",
        "pblancUrl",
    ],
    "kst": [
        "pbanc_sn",
        "intg_pbanc_biz_nm",
        "pbanc_rcpt_bgng_dt",
        "pbanc_rcpt_end_dt",
        "detl_pg_url",
    ],
    "youth": [
        "plcyNo",
        "plcyNm",
        "bizPrdBgngYmd",
        "bizPrdEndYmd",
        "aplyUrlAddr",
    ],
}

# 구조 확인 표에 표시할 상태 메시지입니다.
STATUS_MESSAGES = {
    "missing_file": "raw 파일 없음",
    "json_ok": "JSON 확인 완료",
    "text_needs_check": "XML/TXT 확인 필요",
}


# ============================================================
# 프로젝트 경로 설정
# ============================================================

PROJECT_ROOT = Path.cwd()
RAW_ROOT = PROJECT_ROOT / "data" / "raw"
CLEAN_ROOT = PROJECT_ROOT / "data" / "clean"

# 이번 노트북에서 새로 만들 CSV입니다.
# raw_columns는 "정규화하지 않은 원본 컬럼 통합본"이라는 뜻입니다.
MERGED_RAW_FILE = CLEAN_ROOT / "combined_raw_columns.csv"

# 참고용: 이미 있던 정규화 결과 파일입니다. 이번 노트북의 목표 파일은 아닙니다.
NORMALIZED_FILE = CLEAN_ROOT / "combined.csv"

# API 수집 스크립트 위치입니다.
FETCH_SCRIPT = PROJECT_ROOT / "scripts" / "fetch.py"
APP_PACKAGE = PROJECT_ROOT / "app"


# ============================================================
# helper 함수
# ============================================================

# 파일 경로를 사람이 보기 쉬운 프로젝트 기준 상대경로로 바꿉니다.
def to_project_relative(path):
    """
    예:
    C:/Users/.../PolicyRec/data/raw/biz/a.json
    -> data/raw/biz/a.json
    """
    if path is None:
        return "-"

    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


# source별 raw 폴더에서 가장 최근 파일 1개를 찾습니다
def find_latest_raw_file(source_name):
    """
    source_name 예:
    - "biz"
    - "kst"
    - "youth"

    반환값:
    - 파일을 찾으면 Path 객체
    - 파일이 없으면 None
    """
    source_dir = RAW_ROOT / source_name
    if not source_dir.exists():
        return None

    candidates = []
    for file_pattern in RAW_FILE_PATTERNS:
        candidates.extend(source_dir.glob(file_pattern))

    if not candidates:
        return None

    return max(candidates, key=lambda path: (path.stat().st_mtime, path.name))


# raw 파일을 읽습니다.
def load_raw_file(file_path):
    """
    JSON 파일이면 Python dict/list로 바꿔서 돌려주고,
    XML이나 TXT처럼 구조를 바로 알 수 없는 파일이면 문자열로 돌려줍니다.
    """
    if file_path is None:
        return None, None

    if file_path.suffix.lower() == ".json":
        with file_path.open("r", encoding="utf-8") as file:
            return "json", json.load(file)

    return "text", file_path.read_text(encoding="utf-8", errors="ignore")


# 중첩된 dict/list 안에서 처음 만나는 list를 찾습니다.
def find_first_list(value):
    """
    API마다 목록이 들어 있는 위치가 다릅니다.
    Youthcenter처럼 result 안쪽에 목록이 들어 있는 경우를 처리하기 위해 사용합니다.
    """
    if isinstance(value, list):
        return value

    if isinstance(value, dict):
        for nested_value in value.values():
            found = find_first_list(nested_value)
            if found is not None:
                return found

    return None


# API 응답 전체에서 실제 공고 목록만 꺼냅니다.
def extract_items(source_name, payload):
    """
    반환값은 항상 list입니다.
    데이터를 못 찾으면 빈 list를 돌려줍니다.
    """
    if payload is None:
        return []

    if source_name == "biz":
        if isinstance(payload, dict) and isinstance(payload.get("jsonArray"), list):
            return payload["jsonArray"]
        return find_first_list(payload) or []

    if source_name == "kst":
        if isinstance(payload, dict) and isinstance(payload.get("data"), list):
            return payload["data"]
        return find_first_list(payload) or []

    if source_name == "youth":
        return find_first_list(payload) or []

    return []


# 공고 item에서 보고 싶은 필드만 골라 작은 dict로 만듭니다.
def pick_fields(item, field_names):
    """
    전체 필드를 한 번에 보면 너무 길기 때문에,
    처음에는 제목, 기간, URL처럼 중요한 필드만 확인합니다.
    """
    if not isinstance(item, dict):
        return {"raw_preview": str(item)[:300]}

    return {field_name: item.get(field_name) for field_name in field_names}


# 공고 item 1개의 모든 필드를 세로 표로 바꿉니다.
def make_vertical_table(item):
    """
    컬럼이 많은 데이터를 가로로 보면 읽기 어렵습니다.
    그래서 행과 열을 뒤집어, 필드명과 값을 위에서 아래로 확인합니다.
    """
    if not isinstance(item, dict):
        return pd.DataFrame({"value": [str(item)]}, index=["raw_preview"])

    table = pd.DataFrame([item]).T
    table.columns = ["value"]
    return table


# CSV에 넣기 어려운 dict/list 값을 JSON 문자열로 바꿉니다.
def stringify_nested_value(value):
    """
    CSV는 표 형식이라 한 칸 안에 또 다른 dict/list가 들어가면 다루기 어렵습니다.
    그래서 중첩값만 문자열로 바꾸고, 일반 문자열/숫자/빈값은 그대로 둡니다.
    """
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


# source 하나의 item 목록을 DataFrame으로 바꿉니다.
def make_source_dataframe(source_name, snapshot):
    """
    중요한 원칙:
    - 원본 컬럼명은 바꾸지 않습니다.
    - 어떤 API에서 온 데이터인지 알 수 있도록 _source 관련 컬럼만 앞에 추가합니다.
    """
    items = snapshot.get("items", [])
    file_path = snapshot.get("file_path")

    if not items:
        return pd.DataFrame(columns=METADATA_COLUMNS)

    source_df = pd.DataFrame(items)

    # dict/list처럼 CSV 한 칸에 바로 넣기 어려운 값만 문자열로 바꿉니다.
    for column in source_df.columns:
        source_df[column] = source_df[column].map(stringify_nested_value)

    source_df.insert(0, "_source_row_number", range(1, len(source_df) + 1))
    source_df.insert(0, "_source_file", to_project_relative(file_path))
    source_df.insert(0, "_source_name", SOURCE_LABELS[source_name])
    source_df.insert(0, "_source", source_name)

    return source_df


print("프로젝트 루트:", PROJECT_ROOT)
print("raw 폴더:", RAW_ROOT)
print("통합 CSV 저장 위치:", MERGED_RAW_FILE)
print("대상 source:", SOURCES)

프로젝트 루트: c:\Users\min2m\github\PolicyRec
raw 폴더: c:\Users\min2m\github\PolicyRec\data\raw
통합 CSV 저장 위치: c:\Users\min2m\github\PolicyRec\data\clean\combined_raw_columns.csv
대상 source: ['biz', 'kst', 'youth']


## 이 노트북에서 정의한 함수

아래 함수들은 반복 작업을 줄이기 위해 만든 작은 도구입니다.

| 함수명 | 하는 일 |
| :--- | :--- |
| `to_project_relative()` | 긴 절대경로를 프로젝트 기준 짧은 경로로 바꿉니다. |
| `find_latest_raw_file()` | source별 최신 raw 파일을 찾습니다. |
| `load_raw_file()` | JSON, XML, TXT 파일을 읽습니다. |
| `find_first_list()` | 중첩된 응답 안에서 목록(list)을 찾습니다. |
| `extract_items()` | API 응답에서 실제 공고 목록만 꺼냅니다. |
| `pick_fields()` | 공고 1개에서 보고 싶은 필드만 고릅니다. |
| `make_vertical_table()` | 공고 1개의 전체 필드를 세로 표로 보여줍니다. |
| `stringify_nested_value()` | CSV에 넣기 어려운 dict/list 값을 문자열로 바꿉니다. |
| `make_source_dataframe()` | source 하나를 원본 컬럼 그대로 DataFrame으로 바꿉니다. |

함수를 먼저 만들어 두면, 뒤에서는 복잡한 코드를 반복해서 쓰지 않아도 됩니다.

## 설정값을 위로 모아 둔 이유

아래 값들은 나중에 자주 바뀔 수 있습니다.

- 몇 개 source를 볼지: `SOURCES`
- API를 다시 호출할지: `RUN_FETCH`
- 몇 번째 페이지를 받을지: `PAGE`, `PAGE_SIZE`
- 화면에 몇 개까지 미리 보여줄지: `MAX_TOP_LEVEL_KEYS`, `MAX_SAMPLE_ITEM_KEYS`, `PREVIEW_ROW_COUNT`
- 어떤 대표 필드를 먼저 볼지: `PREVIEW_FIELDS`
- CSV 저장 인코딩: `CSV_ENCODING`

이런 값을 코드 중간중간에 직접 숫자로 넣어 두면 나중에 찾기 어렵습니다. 그래서 첫 번째 코드 셀 위쪽에 모아 두었습니다.

## 1단계. 현재 프로젝트 상태 확인하기

API를 호출하기 전에 먼저 폴더와 파일이 있는지 확인합니다.

특히 `scripts/fetch.py`는 `app` 패키지를 import해서 실행됩니다. 따라서 현재 프로젝트 루트에 `app` 폴더가 없으면 API 재호출은 실패할 수 있습니다. 이 경우에도 이미 저장된 `data/raw` 파일을 읽는 분석 셀은 계속 사용할 수 있습니다.

In [2]:
# 현재 프로젝트에 필요한 폴더와 파일이 있는지 확인합니다.
# 이 표는 "지금 노트북을 어디까지 실행할 수 있는지"를 빠르게 보여줍니다.

status_rows = [
    {
        "check": "프로젝트 루트",
        "exists": PROJECT_ROOT.exists(),
        "path": str(PROJECT_ROOT),
        "memo": "현재 노트북 실행 기준 폴더",
    },
    {
        "check": "raw 데이터 폴더",
        "exists": RAW_ROOT.exists(),
        "path": to_project_relative(RAW_ROOT),
        "memo": "API 원본 응답 저장 위치",
    },
    {
        "check": "수집 스크립트",
        "exists": FETCH_SCRIPT.exists(),
        "path": to_project_relative(FETCH_SCRIPT),
        "memo": "API를 새로 호출할 때 사용",
    },
    {
        "check": "app 패키지",
        "exists": APP_PACKAGE.exists(),
        "path": to_project_relative(APP_PACKAGE),
        "memo": "fetch.py 실행에 필요",
    },
    {
        "check": "원본 컬럼 통합 CSV",
        "exists": MERGED_RAW_FILE.exists(),
        "path": to_project_relative(MERGED_RAW_FILE),
        "memo": "이 노트북에서 새로 만들 결과 파일",
    },
    {
        "check": "기존 정규화 CSV",
        "exists": NORMALIZED_FILE.exists(),
        "path": to_project_relative(NORMALIZED_FILE),
        "memo": "참고용. 이번 목표는 정규화 CSV가 아님",
    },
]

status_df = pd.DataFrame(status_rows)
display(status_df)

,check,exists,path,memo
0,프로젝트 루트,True,c:\Users\min2m\github\PolicyRec,현재 노트북 실행 기준 폴더
1,raw 데이터 폴더,True,data\raw,API 원본 응답 저장 위치
2,수집 스크립트,True,scripts\fetch.py,API를 새로 호출할 때 사용
3,app 패키지,True,app,fetch.py 실행에 필요
4,원본 컬럼 통합 CSV,True,data\clean\combined_raw_columns.csv,이 노트북에서 새로 만들 결과 파일
5,기존 정규화 CSV,True,data\clean\combined.csv,참고용. 이번 목표는 정규화 CSV가 아님


## 2단계. API를 새로 호출할지 결정하기

원본 데이터를 새로 받고 싶다면 첫 번째 코드 셀에서 `RUN_FETCH = True`로 바꾼 뒤 이 셀을 실행합니다.

처음 공부할 때는 API를 매번 호출하지 않는 편이 좋습니다. 이미 저장된 raw 파일을 먼저 읽어도 데이터 구조를 충분히 이해할 수 있기 때문입니다.

In [ ]:
# API 재호출 셀입니다.
# RUN_FETCH가 False이면 아무 것도 새로 받지 않고, 기존 raw 파일을 사용합니다.

if not RUN_FETCH:
    print("RUN_FETCH=False 이므로 API를 
    새로 호출하지 않습니다.")
    print("이미 저장된 data/raw 파일을 사용해 다음 단계를 진행합니다.")
else:
    missing_parts = []

    if not FETCH_SCRIPT.exists():
        missing_parts.append("scripts/fetch.py 파일이 없습니다.")

    if not APP_PACKAGE.exists():
        missing_parts.append("app 폴더가 없습니다. fetch.py가 app.collectors를 import하므로 실행이 어려울 수 있습니다.")

    if missing_parts:
        print("API 재호출을 건너뜁니다.")
        print("이유:")
        for reason in missing_parts:
            print("-", reason)
        print("\n대신 기존 data/raw 파일을 사용해 구조 분석을 계속합니다.")
    else:
        command = [
            sys.executable,
            str(FETCH_SCRIPT),
            "--sources",
            *SOURCES,
            "--page",
            str(PAGE),
            "--page-size",
            str(PAGE_SIZE),
        ]

        if FORCE_FETCH:
            command.append("--force")

        result = subprocess.run(
            command,
            cwd=PROJECT_ROOT,
            text=True,
            capture_output=True,
        )

        print(result.stdout)

        if result.stderr.strip():
            print("----- stderr -----")
            print(result.stderr)

        print("종료 코드:", result.returncode)

RUN_FETCH=False 이므로 API를 새로 호출하지 않습니다.
이미 저장된 data/raw 파일을 사용해 다음 단계를 진행합니다.


## 3단계. raw 파일이 저장되었는지 확인하기

API를 호출했든, 기존 파일을 사용하든, 다음으로 확인할 것은 실제 raw 파일입니다.

이 단계에서 보는 것:

- source별 폴더가 있는지
- 가장 최근 raw 파일 이름이 무엇인지
- 파일 형식이 JSON인지 XML/TXT인지

In [4]:
# source별 최신 raw 파일 1개를 찾아 표로 정리합니다.

latest_files = {}
raw_file_rows = []

for source_name in SOURCES:
    latest_file = find_latest_raw_file(source_name)
    latest_files[source_name] = latest_file

    raw_file_rows.append({
        "source": source_name,
        "api_name": SOURCE_LABELS[source_name],
        "folder": to_project_relative(RAW_ROOT / source_name),
        "latest_file": latest_file.name if latest_file else "파일 없음",
        "file_type": latest_file.suffix.lower() if latest_file else "-",
    })

raw_file_df = pd.DataFrame(raw_file_rows)
display(raw_file_df)

,source,api_name,folder,latest_file,file_type
0,biz,Bizinfo,data\raw\biz,bizinfo_page1_size10_20260421_122751.json,.json
1,kst,K-Startup,data\raw\kst,kstartup_page1_size10_20260421_122753.json,.json
2,youth,Youthcenter,data\raw\youth,youthcenter_page1_size10_20260421_122753.json,.json


## 4단계. 각 API 응답 구조 확인하기

API마다 응답 모양이 다릅니다.

예를 들어:

- Bizinfo는 `jsonArray` 안에 공고 목록이 있습니다.
- K-Startup은 `data` 안에 공고 목록이 있습니다.
- Youthcenter는 `result` 안쪽에 목록이 들어 있습니다.

이 셀은 각 source의 최상위 키와 item 개수를 요약해서 보여줍니다.

In [5]:
# 최신 raw 파일을 읽고, source별 구조를 요약합니다.

raw_snapshots = {}
structure_rows = []

for source_name in SOURCES:
    latest_file = latest_files.get(source_name)
    file_type, payload = load_raw_file(latest_file)
    items = extract_items(source_name, payload) if file_type == "json" else []

    raw_snapshots[source_name] = {
        "file_path": latest_file,
        "file_type": file_type,
        "payload": payload,
        "items": items,
    }

    if latest_file is None:
        structure_rows.append({
            "source": source_name,
            "status": STATUS_MESSAGES["missing_file"],
            "top_level_keys": [],
            "item_count": 0,
            "sample_item_keys": [],
        })
        continue

    if file_type == "json":
        top_level_keys = list(payload.keys())[:MAX_TOP_LEVEL_KEYS] if isinstance(payload, dict) else []
        sample_item_keys = list(items[0].keys())[:MAX_SAMPLE_ITEM_KEYS] if items and isinstance(items[0], dict) else []

        structure_rows.append({
            "source": source_name,
            "status": STATUS_MESSAGES["json_ok"],
            "top_level_keys": top_level_keys,
            "item_count": len(items),
            "sample_item_keys": sample_item_keys,
        })
    else:
        preview = payload[:TEXT_PREVIEW_LENGTH].replace("\n", " ") if isinstance(payload, str) else ""
        structure_rows.append({
            "source": source_name,
            "status": STATUS_MESSAGES["text_needs_check"],
            "top_level_keys": [preview],
            "item_count": 0,
            "sample_item_keys": [],
        })

structure_df = pd.DataFrame(structure_rows)
display(structure_df)

,source,status,top_level_keys,item_count,sample_item_keys
0,biz,JSON 확인 완료,[jsonArray],10,"[trgetNm, updtPnttm, hashtags, inqireCo, creat..."
1,kst,JSON 확인 완료,"[currentCount, data, matchCount, page, perPage...",10,"[aply_excl_trgt_ctnt, aply_mthd_eml_rcpt_istc,..."
2,youth,JSON 확인 완료,"[resultCode, resultMessage, result]",10,"[plcyNo, bscPlanCycl, bscPlanPlcyWayNo, bscPla..."


## 5단계. source별 대표 item 1개만 먼저 보기

전체 필드를 처음부터 다 보면 너무 길어서 이해하기 어렵습니다.

먼저 각 source에서 제목, 기간, URL처럼 눈에 잘 들어오는 필드만 확인합니다. 여기서도 컬럼명을 바꾸지는 않습니다.

대표 필드 목록은 첫 번째 코드 셀의 `PREVIEW_FIELDS`에서 바꿀 수 있습니다.

In [6]:
# source별 대표 item 1개를 간단히 확인합니다.

for source_name in SOURCES:
    snapshot = raw_snapshots[source_name]
    items = snapshot["items"]

    print(f"\n===== {SOURCE_LABELS[source_name]} 대표 item =====")

    if not items:
        print("보여 줄 item이 없습니다. 파일이 없거나 JSON 구조를 더 확인해야 합니다.")
        continue

    first_item = items[0]
    preview = pick_fields(first_item, PREVIEW_FIELDS[source_name])

    display(pd.DataFrame([preview]))
    print("전체 키 일부:", list(first_item.keys())[:MAX_SAMPLE_ITEM_KEYS])


===== Bizinfo 대표 item =====


,pblancId,pblancNm,reqstBeginEndDe,jrsdInsttNm,pblancUrl
0,PBLN_000000000121046,[서남권] 2026년 지역혁신클러스터육성(비R&D) 기업지원사업 모집 통합 공고,2026-04-14 ~ 2026-05-06,전라남도,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...


전체 키 일부: ['trgetNm', 'updtPnttm', 'hashtags', 'inqireCo', 'creatPnttm', 'pblancNm', 'pblancId', 'printFlpthNm', 'refrncNm', 'rceptEngnHmpgUrl', 'fileNm', 'pblancUrl', 'jrsdInsttNm', 'excInsttNm', 'totCnt']

===== K-Startup 대표 item =====


,pbanc_sn,intg_pbanc_biz_nm,pbanc_rcpt_bgng_dt,pbanc_rcpt_end_dt,detl_pg_url
0,177302,2026년 3D-FAB 크라우드펀딩 지원사업 참여 기업 모집,20260420,20260515,https://www.k-startup.go.kr/web/contents/bizpb...


전체 키 일부: ['aply_excl_trgt_ctnt', 'aply_mthd_eml_rcpt_istc', 'aply_mthd_etc_istc', 'aply_mthd_fax_rcpt_istc', 'aply_mthd_onli_rcpt_istc', 'aply_mthd_pssr_rcpt_istc', 'aply_mthd_vst_rcpt_istc', 'aply_trgt', 'aply_trgt_ctnt', 'biz_aply_url', 'biz_enyy', 'biz_gdnc_url', 'biz_pbanc_nm', 'biz_prch_dprt_nm', 'biz_trgt_age']

===== Youthcenter 대표 item =====


,plcyNo,plcyNm,bizPrdBgngYmd,bizPrdEndYmd,aplyUrlAddr
0,20260421005400112773,미소금융 청년 미래이음 대출,,,https://www.kinfa.or.kr/financialProduct/young...


전체 키 일부: ['plcyNo', 'bscPlanCycl', 'bscPlanPlcyWayNo', 'bscPlanFcsAsmtNo', 'bscPlanAsmtNo', 'pvsnInstGroupCd', 'plcyPvsnMthdCd', 'plcyAprvSttsCd', 'plcyNm', 'plcyKywdNm', 'plcyExplnCn', 'lclsfNm', 'mclsfNm', 'plcySprtCn', 'sprvsnInstCd']


## 6단계. 대표 item의 모든 필드 보기

이제 각 source에서 첫 번째 item의 전체 필드를 봅니다.

이 단계의 목적은 “원본에는 어떤 컬럼들이 있는지”를 확인하는 것입니다. 표를 세로로 출력하면 필드가 많아도 한 줄씩 읽기 쉽습니다.

In [7]:
# 각 source에서 item 1개를 골라 모든 필드를 세로로 출력합니다.

for source_name in SOURCES:
    snapshot = raw_snapshots[source_name]
    items = snapshot["items"]

    print(f"\n{'=' * 20} {SOURCE_LABELS[source_name]} 상세 분석 {'=' * 20}")

    if not items:
        print("보여 줄 item이 없습니다. 데이터 구조를 다시 확인하세요.")
        continue

    first_item = items[0]
    full_view = make_vertical_table(first_item)

    display(full_view)
    print(f"총 필드 개수: {len(first_item)}개")


==================== Bizinfo 상세 분석 ====================


,value
trgetNm,중소기업
updtPnttm,2026-04-20 16:19:15
hashtags,"기술,광주,전남,기술 사업화,2026,전남지역산업진흥원,광주지역산업진흥원,녹색에너지..."
inqireCo,1289
creatPnttm,2026-04-20 16:14:47
pblancNm,[서남권] 2026년 지역혁신클러스터육성(비R&D) 기업지원사업 모집 통합 공고
pblancId,PBLN_000000000121046
printFlpthNm,https://www.bizinfo.go.kr/cmm/fms/getImageFile...
refrncNm,(전남지역산업진흥원) 061-339-9722 (전남테크노파크) 061-337-504...
rceptEngnHmpgUrl,https://data.jntp.or.kr/jntp/content/business/...


총 필드 개수: 22개

==================== K-Startup 상세 분석 ====================


,value
aply_excl_trgt_ctnt,"기관, 지자체 등 기타 기관을 통해 동일한 사업 콘텐츠로 크라우드펀딩 관련 지원을 ..."
aply_mthd_eml_rcpt_istc,None
aply_mthd_etc_istc,None
aply_mthd_fax_rcpt_istc,None
aply_mthd_onli_rcpt_istc,https://3d-fab.kr/kor/event/view.php?pNo=1&idx=26
aply_mthd_pssr_rcpt_istc,None
aply_mthd_vst_rcpt_istc,None
aply_trgt,"일반인,대학,연구기관,일반기업,1인 창조기업"
aply_trgt_ctnt,"3D프린팅 기술을 활용하여 개발/제작한 제품을 출시하였거나, 출시 예정인 기업 \r..."
biz_aply_url,None


총 필드 개수: 30개

==================== Youthcenter 상세 분석 ====================


,value
plcyNo,20260421005400112773
bscPlanCycl,2
bscPlanPlcyWayNo,004
bscPlanFcsAsmtNo,011
bscPlanAsmtNo,039
pvsnInstGroupCd,0054001
plcyPvsnMthdCd,0042003
plcyAprvSttsCd,0044002
plcyNm,미소금융 청년 미래이음 대출
plcyKywdNm,"금리혜택,대출"


총 필드 개수: 60개


## 7단계. 데이터를 하나의 CSV로 통합하기

여기서는 **정규화하지 않습니다.**

즉, `pblancNm`을 `title`로 바꾸거나, `pbanc_rcpt_bgng_dt`를 `apply_start`로 바꾸지 않습니다. 각 API가 준 원본 컬럼명을 그대로 유지합니다.

다만 세 데이터를 한 파일에 붙이면 어느 행이 어디서 왔는지 헷갈릴 수 있으므로, 아래 관리용 컬럼만 앞에 추가합니다.

- `_source`: `biz`, `kst`, `youth`
- `_source_name`: 사람이 읽기 쉬운 API 이름
- `_source_file`: 이 행이 나온 raw 파일
- `_source_row_number`: source 안에서 몇 번째 item인지

In [8]:
# source별 DataFrame을 만들고, 세로 방향으로 하나로 붙입니다.
# sort=False를 주면 pandas가 컬럼을 이름순으로 다시 정렬하지 않고, 등장 순서를 최대한 유지합니다.

source_frames = []
merge_summary_rows = []

for source_name in SOURCES:
    source_df = make_source_dataframe(source_name, raw_snapshots[source_name])
    source_frames.append(source_df)

    original_columns = [column for column in source_df.columns if column not in METADATA_COLUMNS]
    merge_summary_rows.append({
        "source": source_name,
        "api_name": SOURCE_LABELS[source_name],
        "row_count": len(source_df),
        "original_column_count": len(original_columns),
        "raw_file": source_df["_source_file"].iloc[0] if len(source_df) else "-",
    })

raw_merged_df = pd.concat(source_frames, ignore_index=True, sort=False)

# data/clean 폴더가 없으면 만들어 둡니다.
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

# Excel에서 한글이 깨지는 일을 줄이기 위해 utf-8-sig로 저장합니다.
raw_merged_df.to_csv(MERGED_RAW_FILE, index=False, encoding=CSV_ENCODING)

print("원본 컬럼 통합 CSV 저장 완료:", to_project_relative(MERGED_RAW_FILE))
print("통합 결과 행/열 개수:", raw_merged_df.shape)

print("\nsource별 통합 요약:")
display(pd.DataFrame(merge_summary_rows))

print("\n통합 데이터 미리보기:")
display(raw_merged_df.head(PREVIEW_ROW_COUNT))

원본 컬럼 통합 CSV 저장 완료: data\clean\combined_raw_columns.csv
통합 결과 행/열 개수: (30, 116)

source별 통합 요약:


,source,api_name,row_count,original_column_count,raw_file
0,biz,Bizinfo,10,22,data\raw\biz\bizinfo_page1_size10_20260421_122...
1,kst,K-Startup,10,30,data\raw\kst\kstartup_page1_size10_20260421_12...
2,youth,Youthcenter,10,60,data\raw\youth\youthcenter_page1_size10_202604...



통합 데이터 미리보기:


,_source,_source_name,_source_file,_source_row_number,trgetNm,updtPnttm,hashtags,inqireCo,creatPnttm,pblancNm,...,rgtrHghrkInstCd,rgtrHghrkInstCdNm,zipCd,plcyMajorCd,jobCd,schoolCd,aplyYmd,frstRegDt,lastMdfcnDt,sbizCd
0,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,1,중소기업,2026-04-20 16:19:15,"기술,광주,전남,기술 사업화,2026,전남지역산업진흥원,광주지역산업진흥원,녹색에너지...",1289.0,2026-04-20 16:14:47,[서남권] 2026년 지역혁신클러스터육성(비R&D) 기업지원사업 모집 통합 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,2,중소기업,2026-04-20 16:33:59,"기술,서울,부산,대구,인천,광주,대전,울산,세종,경기,강원,충북,충남,전북,전남,경...",1204.0,2026-04-20 15:47:43,2026년 2차 정보보호핵심원천기술개발사업 신규지원 대상과제 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,3,중소기업,2026-04-20 16:19:15,"기술,경영,충북,충북바이오산학융합원,충청북도,2026,바이오기업,중소기업,의약품,의...",1137.0,2026-04-20 15:46:40,[충북] 2026년 바이오기업 GMP인증 지원사업 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,4,중소기업,2026-04-20 16:19:15,"수출,서울,부산,대구,인천,광주,대전,울산,세종,경기,강원,충북,충남,전북,전남,경...",1303.0,2026-04-20 15:40:52,2026년 K-EXPO USA K-뷰티 팝업 스토어 참여기업 모집 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,5,창업벤처,2026-04-20 16:19:15,"기술,창업,충북,충북바이오산학융합원,충청북도,2026,벤처기업,스타트업,공동연구장비...",1102.0,2026-04-20 15:37:50,[충북] 2026년 벤처 및 스타트업 기업 공동연구장비 활용 지원사업 참여기업 모집 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,6,중소기업,2026-04-20 16:19:15,"수출,서울,부산,대구,인천,광주,대전,울산,세종,경기,강원,충북,충남,전북,전남,경...",1205.0,2026-04-20 15:31:49,2026년 K테크서비스 중소벤처기업 사우디 파트너십 구축 지원사업 참여기업 모집 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,7,창업벤처,2026-04-20 16:19:15,"창업,대전,2026,대전창조경제혁신센터,대전광역시,소셜벤처,예비 창업기업,기후테크,...",1099.0,2026-04-20 15:30:24,[대전] 2026년 소셜 임팩트 랩(Lab) 예비 창업기업 모집 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,8,중소기업,2026-04-20 16:19:15,"인력,충북,2026,임차기숙사,기숙사임차비,괴산군,기숙사,충청북도,충청북도기업진흥원...",1078.0,2026-04-20 15:29:16,[충북] 2026년 2차 기숙사임차비지원사업 참여기업 추가모집 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,9,중소기업,2026-04-20 16:19:15,"기술,서울,부산,대구,인천,광주,대전,울산,세종,경기,강원,충북,충남,전북,전남,경...",1078.0,2026-04-20 15:19:39,2026년 IoT 보안인증 인증시험 수수료 지원 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,biz,Bizinfo,data\raw\biz\bizinfo_page1_size10_20260421_122...,10,중소기업,2026-04-20 16:19:15,"경영,서울,부산,대구,인천,광주,대전,울산,세종,경기,강원,충북,충남,전북,전남,경...",987.0,2026-04-20 15:18:02,2026년 경북 원자력 선도기업 육성사업 지원기업 모집 공고,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8단계. 통합 CSV 구조 확인하기

세 API의 원본 컬럼을 그대로 붙였기 때문에 빈칸이 많이 생기는 것은 정상입니다.

예를 들어 Bizinfo 행에는 K-Startup 전용 컬럼인 `pbanc_sn`이 비어 있고, K-Startup 행에는 Bizinfo 전용 컬럼인 `pblancId`가 비어 있을 수 있습니다.

이 방식의 장점은 원본 정보를 잃지 않는 것입니다. 나중에 추천이나 요약에 필요한 컬럼이 확정되면, 그때 별도의 정규화 CSV를 만들면 됩니다.

In [9]:
# 통합 CSV의 컬럼과 source별 빈값 개수를 간단히 확인합니다.

print("통합 CSV 파일:", to_project_relative(MERGED_RAW_FILE))
print("전체 행/열:", raw_merged_df.shape)

print("\n전체 컬럼 목록:")
column_info_df = pd.DataFrame({
    "column_order": range(1, len(raw_merged_df.columns) + 1),
    "column_name": raw_merged_df.columns,
})
display(column_info_df)

print("\nsource별 행 개수:")
display(raw_merged_df["_source"].value_counts().rename_axis("source").reset_index(name="row_count"))

print("\nsource별로 값이 들어 있는 원본 컬럼 개수:")
non_empty_rows = []
for source_name in SOURCES:
    source_only_df = raw_merged_df[raw_merged_df["_source"] == source_name]
    original_columns = [column for column in raw_merged_df.columns if column not in METADATA_COLUMNS]
    non_empty_count = source_only_df[original_columns].notna().any(axis=0).sum()

    non_empty_rows.append({
        "source": source_name,
        "api_name": SOURCE_LABELS[source_name],
        "non_empty_original_columns": int(non_empty_count),
    })

display(pd.DataFrame(non_empty_rows))

통합 CSV 파일: data\clean\combined_raw_columns.csv
전체 행/열: (30, 116)

전체 컬럼 목록:


,column_order,column_name
0,1,_source
1,2,_source_name
2,3,_source_file
3,4,_source_row_number
4,5,trgetNm
...,...,...
111,112,schoolCd
112,113,aplyYmd
113,114,frstRegDt
114,115,lastMdfcnDt



source별 행 개수:


,source,row_count
0,biz,10
1,kst,10
2,youth,10



source별로 값이 들어 있는 원본 컬럼 개수:


,source,api_name,non_empty_original_columns
0,biz,Bizinfo,22
1,kst,K-Startup,26
2,youth,Youthcenter,60


## 정리와 다음 작업

현재 `PolicyRec_v1.0.ipynb`는 **v1 단계**를 담당합니다.

v1에서 완료되는 것:

1. raw 데이터 위치 확인
2. Bizinfo, K-Startup, Youthcenter 원본 JSON 구조 확인
3. source별 대표 item 확인
4. 각 source의 원본 컬럼을 유지한 DataFrame 생성
5. `data/clean/combined_raw_columns.csv` 저장

v1에서 하지 않는 것:

- 통합 스키마 (ex.`pblancNm`, `intg_pbanc_biz_nm`, `plcyNm`을 하나의 `title` 컬럼으로 합치기)
- 날짜 형식을 강제로 통일하기
- 결측값을 임의 기준으로 채우기
- 첨부파일 본문을 읽어서 붙이기

다음 단계는 `v1.n`입니다.

v1.n에서는 v1 결과를 기반으로 최소 정규화를 진행합니다. 이때 컬럼명 통일, 날짜 형식 통일, 결측값 처리 기준을 정합니다.

그 다음 단계는 `v2`입니다.

v2에서는 공고 목록 데이터에 첨부파일 정보를 추가합니다. 첨부파일은 PDF, HWP, ZIP처럼 형식이 다양해서 예외가 많으므로, v1.n에서 기본 목록 데이터 구조가 어느 정도 안정된 뒤 진행하는 것이 좋습니다.